# 15 - WLASL2000 Full Landmark Extraction

## Goal

Process **all ~12K available WLASL2000 videos** through MediaPipe hand landmark
extraction. This creates the landmark dataset needed for progressive training
(100 -> 300 -> 1000 -> 2000 classes).

## Key Features

- **Resume capability**: If interrupted, re-running picks up from the last checkpoint
- **Periodic checkpoints**: Progress saved every 500 videos
- **Error logging**: All failures logged separately for inspection
- **Robust error handling**: Corrupted videos and detection failures handled gracefully

## Estimates

- **Time**: ~4.3 hours (11,980 videos at ~1.3s each)
- **Disk**: ~330 MB (landmark .npy files)
- **Output**: `06_datasets/wlasl/landmarks/wlasl2000/`

## Pipeline

```
MP4 video -> OpenCV (sample 64 frames) -> MediaPipe (21 landmarks x 3 coords)
          -> Interpolate missing -> Normalize (wrist-centered) -> Save .npy
```

In [1]:
# === 1. Imports and Configuration ===

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request
import os
import cv2
import numpy as np
import json
import time
from pathlib import Path
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional, Set
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# === Configuration ===
MAX_FRAMES = 64
MIN_DETECTION_RATE = 0.3  # Reject videos with < 30% hand detection
CHECKPOINT_EVERY = 500    # Save progress every N videos

# === Paths ===
BASE = Path("/home/fransando/Dev/tfg-sign-language-recognition")
KAGGLE_ROOT = BASE / "06_datasets" / "wlasl" / "raw" / "kaggle_wlasl"
VIDEOS_DIR = KAGGLE_ROOT / "videos"
NSLT_2000_PATH = KAGGLE_ROOT / "nslt_2000.json"
CLASS_LIST_PATH = KAGGLE_ROOT / "wlasl_class_list.txt"

OUTPUT_ROOT = BASE / "06_datasets" / "wlasl" / "landmarks" / "wlasl2000"
PROGRESS_FILE = OUTPUT_ROOT / "_progress.json"
ERROR_LOG_FILE = OUTPUT_ROOT / "_errors.json"
MODEL_PATH = BASE / "hand_landmarker.task"

print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Output: {OUTPUT_ROOT}")
print(f"Checkpoint every: {CHECKPOINT_EVERY} videos")
print(f"Max frames: {MAX_FRAMES}")
print(f"Min detection rate: {MIN_DETECTION_RATE}")

Date: 2026-02-18 23:27
Output: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl2000
Checkpoint every: 500 videos
Max frames: 64
Min detection rate: 0.3


In [2]:
# === 2. Load Annotations and Build Processing Manifest ===

# Load nslt_2000 annotations
with open(NSLT_2000_PATH) as f:
    nslt_2000 = json.load(f)
print(f"nslt_2000 entries: {len(nslt_2000)}")

# Load class list (index -> gloss name)
class_list = {}
with open(CLASS_LIST_PATH) as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            class_list[int(parts[0])] = parts[1]
print(f"Class list: {len(class_list)} glosses")

# Build manifest: list of videos to process (only those on disk)
manifest = []
skipped_missing = 0

for video_id, info in nslt_2000.items():
    video_path = VIDEOS_DIR / f"{video_id}.mp4"
    if not video_path.exists():
        skipped_missing += 1
        continue
    
    cls_idx = info["action"][0]
    gloss = class_list.get(cls_idx, f"unknown_{cls_idx}")
    split = info["subset"]
    
    manifest.append({
        "video_id": video_id,
        "gloss": gloss,
        "split": split,
        "class_idx": cls_idx,
    })

# Sort for deterministic ordering
manifest.sort(key=lambda x: x["video_id"])

print(f"\nManifest built:")
print(f"  Total available: {len(manifest)} videos")
print(f"  Skipped (missing): {skipped_missing} videos")

# Split counts
from collections import Counter
split_counts = Counter(m["split"] for m in manifest)
print(f"  Train: {split_counts.get('train', 0)}")
print(f"  Val:   {split_counts.get('val', 0)}")
print(f"  Test:  {split_counts.get('test', 0)}")

# Unique glosses
unique_glosses = set(m["gloss"] for m in manifest)
print(f"  Unique glosses: {len(unique_glosses)}")

# Estimated time
est_seconds = len(manifest) * 1.3
print(f"\nEstimated processing time: {est_seconds/3600:.1f} hours")

nslt_2000 entries: 21095
Class list: 2000 glosses

Manifest built:
  Total available: 11980 videos
  Skipped (missing): 9115 videos
  Train: 8313
  Val:   2253
  Test:  1414
  Unique glosses: 2000

Estimated processing time: 4.3 hours


In [3]:
# === 3. Resume Capability ===

def load_progress(progress_file: Path) -> Tuple[Set[str], Dict]:
    """Load progress from checkpoint file."""
    if not progress_file.exists():
        return set(), {
            "total": 0, "success": 0, "failed": 0,
            "errors": {},
            "detection_rates": [],
            "by_split": {"train": 0, "val": 0, "test": 0},
        }
    with open(progress_file) as f:
        data = json.load(f)
    return set(data["completed_ids"]), data["stats"]


def save_progress(progress_file: Path, completed_ids: Set[str], stats: Dict):
    """Save checkpoint atomically (write to tmp, then rename)."""
    progress_file.parent.mkdir(parents=True, exist_ok=True)
    tmp = progress_file.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        # Don't save detection_rates in checkpoint (too large), only summary stats
        save_stats = stats.copy()
        n_rates = len(save_stats.get("detection_rates", []))
        if n_rates > 0:
            rates = save_stats["detection_rates"]
            save_stats["detection_rate_summary"] = {
                "count": n_rates,
                "mean": float(np.mean(rates)),
                "min": float(np.min(rates)),
                "max": float(np.max(rates)),
            }
        save_stats.pop("detection_rates", None)
        json.dump({"completed_ids": list(completed_ids), "stats": save_stats}, f)
    tmp.rename(progress_file)


def load_error_log(error_log_file: Path) -> List[Dict]:
    """Load existing error log."""
    if not error_log_file.exists():
        return []
    with open(error_log_file) as f:
        return json.load(f)


def save_error_log(error_log_file: Path, errors: List[Dict]):
    """Save error log."""
    error_log_file.parent.mkdir(parents=True, exist_ok=True)
    with open(error_log_file, 'w') as f:
        json.dump(errors, f, indent=2)


# Load existing progress
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
completed_ids, global_stats = load_progress(PROGRESS_FILE)
error_log = load_error_log(ERROR_LOG_FILE)

# Initialize detection_rates list (not loaded from checkpoint to save space)
if "detection_rates" not in global_stats:
    global_stats["detection_rates"] = []

# Filter manifest to remaining work
remaining = [m for m in manifest if m["video_id"] not in completed_ids]

print(f"Resume status:")
print(f"  Already processed: {len(completed_ids)}")
print(f"  Remaining: {len(remaining)}")
if len(completed_ids) > 0:
    print(f"  Success so far: {global_stats['success']}")
    print(f"  Failed so far: {global_stats['failed']}")
    est_remaining = len(remaining) * 1.3
    print(f"  Estimated time remaining: {est_remaining/3600:.1f} hours")

Resume status:
  Already processed: 6000
  Remaining: 5980
  Success so far: 5949
  Failed so far: 51
  Estimated time remaining: 2.2 hours


In [4]:
# === 4. Initialize MediaPipe ===

# Download model if needed
if not MODEL_PATH.exists():
    print("Downloading hand_landmarker.task...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, str(MODEL_PATH))
    print("Model downloaded.")

# Create detector
base_options = python.BaseOptions(model_asset_path=str(MODEL_PATH))
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)
detector = vision.HandLandmarker.create_from_options(options)
print("MediaPipe HandLandmarker initialized.")

MediaPipe HandLandmarker initialized.


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1771453670.189910    5917 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1771453670.205975    5923 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [5]:
# === 5. Processing Functions ===
# (Self-contained, adapted from notebook 08)


def load_video_frames(video_path: Path, max_frames: int = 64) -> List[np.ndarray]:
    """
    Load frames from an MP4 video.
    Uniformly samples if the video has more than max_frames.
    """
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return []

    # Select frame indices
    if total_frames > max_frames:
        indices = set(np.linspace(0, total_frames - 1, max_frames, dtype=int))
    else:
        indices = set(range(total_frames))

    frames = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx in indices:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_rgb)
        frame_idx += 1

    cap.release()
    return frames


def extract_landmarks_from_frame(
    frame: np.ndarray, hands_detector
) -> Optional[np.ndarray]:
    """
    Extract hand landmarks from a single frame using MediaPipe.
    Returns: array (21, 3) or None if no hand detected.
    """
    try:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        result = hands_detector.detect(mp_image)
        if result.hand_landmarks:
            landmarks = result.hand_landmarks[0]
            coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])
            return coords
    except Exception:
        pass  # Treat MediaPipe errors as no detection
    return None


def interpolate_missing_frames(
    landmarks_list: List[Optional[np.ndarray]],
) -> np.ndarray:
    """
    Interpolate frames where no hand was detected.
    Strategy: linear interpolation between valid frames,
    copy nearest valid frame at edges.
    """
    n_frames = len(landmarks_list)
    result = np.zeros((n_frames, 21, 3))

    valid_indices = [i for i, lm in enumerate(landmarks_list) if lm is not None]

    if len(valid_indices) == 0:
        return result

    # Copy valid frames
    for i in valid_indices:
        result[i] = landmarks_list[i]

    # Interpolate missing frames
    for i in range(n_frames):
        if landmarks_list[i] is not None:
            continue

        # Find nearest valid before
        prev_valid = None
        for j in range(i - 1, -1, -1):
            if landmarks_list[j] is not None:
                prev_valid = j
                break

        # Find nearest valid after
        next_valid = None
        for j in range(i + 1, n_frames):
            if landmarks_list[j] is not None:
                next_valid = j
                break

        if prev_valid is not None and next_valid is not None:
            alpha = (i - prev_valid) / (next_valid - prev_valid)
            result[i] = (1 - alpha) * landmarks_list[prev_valid] + alpha * landmarks_list[next_valid]
        elif prev_valid is not None:
            result[i] = landmarks_list[prev_valid]
        elif next_valid is not None:
            result[i] = landmarks_list[next_valid]

    return result


def normalize_landmarks(landmarks: np.ndarray) -> np.ndarray:
    """
    Normalize landmarks to be position/scale invariant.
    Centers on wrist (landmark 0), scales by wrist-to-middle-finger distance.
    """
    normalized = np.copy(landmarks)

    for t in range(landmarks.shape[0]):
        frame = landmarks[t]
        wrist = frame[0]
        frame_centered = frame - wrist

        scale = np.linalg.norm(frame[12] - frame[0])
        if scale > 1e-6:
            frame_normalized = frame_centered / scale
        else:
            frame_normalized = frame_centered

        normalized[t] = frame_normalized

    return normalized


def process_single_video(
    video_path: Path,
    hands_detector,
    max_frames: int = 64,
    min_detection_rate: float = 0.3,
) -> Tuple[Optional[np.ndarray], Dict]:
    """
    Process a single video end-to-end:
    load -> extract landmarks -> interpolate -> normalize.

    Returns:
        landmarks: array (T, 21, 3) or None on failure
        stats: dict with processing statistics
    """
    stats = {
        "success": False,
        "error": None,
        "sampled_frames": 0,
        "detected_frames": 0,
        "detection_rate": 0.0,
    }

    # Load frames
    frames = load_video_frames(video_path, max_frames)
    if len(frames) == 0:
        stats["error"] = "no_frames_loaded"
        return None, stats

    stats["sampled_frames"] = len(frames)

    # Extract landmarks per frame
    landmarks_list = []
    for frame in frames:
        lm = extract_landmarks_from_frame(frame, hands_detector)
        landmarks_list.append(lm)
        if lm is not None:
            stats["detected_frames"] += 1

    stats["detection_rate"] = stats["detected_frames"] / stats["sampled_frames"]

    # Check minimum detection rate
    if stats["detection_rate"] < min_detection_rate:
        stats["error"] = f"low_detection_rate_{stats['detection_rate']:.2f}"
        return None, stats

    # Interpolate missing frames
    landmarks_interpolated = interpolate_missing_frames(landmarks_list)

    # Normalize
    landmarks_normalized = normalize_landmarks(landmarks_interpolated)

    stats["success"] = True
    return landmarks_normalized, stats


print("Processing functions defined.")

Processing functions defined.


---
## Main Processing Loop

This cell processes all remaining videos. It can be re-run safely after
interruption - it will resume from the last checkpoint.

---

In [6]:
# === 6. Main Processing Loop ===

if len(remaining) == 0:
    print("All videos already processed! Nothing to do.")
else:
    print(f"Processing {len(remaining)} videos...")
    print(f"Checkpoint every {CHECKPOINT_EVERY} videos")
    print(f"Estimated time: {len(remaining) * 1.3 / 3600:.1f} hours\n")

    start_time = time.time()

    for i, entry in enumerate(tqdm(remaining, desc="Extracting landmarks")):
        video_id = entry["video_id"]
        gloss = entry["gloss"]
        split = entry["split"]
        video_path = VIDEOS_DIR / f"{video_id}.mp4"

        try:
            landmarks, stats = process_single_video(
                video_path, detector, MAX_FRAMES, MIN_DETECTION_RATE
            )
        except Exception as e:
            stats = {"success": False, "error": f"exception: {str(e)}",
                     "detection_rate": 0.0}
            landmarks = None
            error_log.append({
                "video_id": video_id, "gloss": gloss, "error": str(e)
            })

        global_stats["total"] += 1

        if stats.get("success"):
            # Save landmarks
            output_dir = OUTPUT_ROOT / split / gloss
            output_dir.mkdir(parents=True, exist_ok=True)
            np.save(output_dir / f"{video_id}.npy", landmarks)

            global_stats["success"] += 1
            global_stats["by_split"][split] = global_stats["by_split"].get(split, 0) + 1
            global_stats["detection_rates"].append(stats["detection_rate"])
        else:
            global_stats["failed"] += 1
            err = stats.get("error", "unknown")
            global_stats["errors"][err] = global_stats["errors"].get(err, 0) + 1
            if stats.get("error") and "exception" in str(stats["error"]):
                error_log.append({
                    "video_id": video_id, "gloss": gloss,
                    "error": stats["error"]
                })

        completed_ids.add(video_id)

        # Periodic checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            save_progress(PROGRESS_FILE, completed_ids, global_stats)
            save_error_log(ERROR_LOG_FILE, error_log)
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            remaining_count = len(remaining) - (i + 1)
            eta_seconds = remaining_count / rate if rate > 0 else 0
            tqdm.write(
                f"\n[Checkpoint {i+1}/{len(remaining)}] "
                f"{global_stats['success']} ok, {global_stats['failed']} failed | "
                f"ETA: {eta_seconds/60:.0f} min"
            )

    # Final save
    save_progress(PROGRESS_FILE, completed_ids, global_stats)
    save_error_log(ERROR_LOG_FILE, error_log)

    elapsed_total = time.time() - start_time
    print(f"\nProcessing complete in {elapsed_total/3600:.1f} hours")

Processing 5980 videos...
Checkpoint every 500 videos
Estimated time: 2.2 hours



Extracting landmarks:   8%|▊         | 500/5980 [11:39<2:20:05,  1.53s/it]


[Checkpoint 500/5980] 6442 ok, 58 failed | ETA: 128 min


Extracting landmarks:  17%|█▋        | 1000/5980 [23:13<2:05:01,  1.51s/it]


[Checkpoint 1000/5980] 6939 ok, 61 failed | ETA: 116 min


...


[Checkpoint 5000/5980] 10882 ok, 118 failed | ETA: 23 min


Extracting landmarks:  92%|█████████▏| 5500/5980 [2:05:36<09:11,  1.15s/it]


[Checkpoint 5500/5980] 11371 ok, 129 failed | ETA: 11 min


Extracting landmarks:  96%|█████████▋| 5766/5980 [2:11:04<05:51,  1.64s/it][h264 @ 0x5d26499b2440] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x5d26499b2440] missing picture in access unit with size 10780
[h264 @ 0x5d2649c5c580] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x5d2649c5c580] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5d2649b5a100] stream 1, offset 0x2a27a7: partial file
Extracting landmarks: 100%|██████████| 5980/5980 [2:16:54<00:00,  1.37s/it]


Processing complete in 2.3 hours


In [7]:
# === 7. Processing Report ===

print("=" * 70)
print("PROCESSING REPORT")
print("=" * 70)

print(f"\nTotal processed: {global_stats['total']}")
print(f"  Successful: {global_stats['success']} "
      f"({100 * global_stats['success'] / max(global_stats['total'], 1):.1f}%)")
print(f"  Failed: {global_stats['failed']} "
      f"({100 * global_stats['failed'] / max(global_stats['total'], 1):.1f}%)")

print(f"\nBy split:")
for split, count in sorted(global_stats["by_split"].items()):
    print(f"  {split}: {count}")

if global_stats["detection_rates"]:
    rates = np.array(global_stats["detection_rates"])
    print(f"\nDetection rate (successful videos):")
    print(f"  Mean: {rates.mean():.1%}")
    print(f"  Min: {rates.min():.1%}")
    print(f"  Max: {rates.max():.1%}")
    print(f"  Median: {np.median(rates):.1%}")

if global_stats["errors"]:
    print(f"\nError breakdown:")
    for err, count in sorted(global_stats["errors"].items(), key=lambda x: -x[1])[:10]:
        print(f"  {err}: {count}")
    if len(global_stats["errors"]) > 10:
        print(f"  ... and {len(global_stats['errors']) - 10} more error types")

PROCESSING REPORT

Total processed: 11980
  Successful: 11844 (98.9%)
  Failed: 136 (1.1%)

By split:
  test: 1402
  train: 8210
  val: 2232

Detection rate (successful videos):
  Mean: 68.0%
  Min: 31.2%
  Max: 100.0%
  Median: 65.6%

Error breakdown:
  low_detection_rate_0.30: 38
  low_detection_rate_0.28: 27
  low_detection_rate_0.27: 23
  low_detection_rate_0.25: 12
  low_detection_rate_0.22: 10
  low_detection_rate_0.20: 6
  low_detection_rate_0.23: 4
  low_detection_rate_0.16: 4
  low_detection_rate_0.14: 3
  low_detection_rate_0.17: 3
  ... and 4 more error types


In [8]:
# === 8. Generate metadata.json ===

# Count .npy files per gloss per split
per_gloss_counts = {}
total_npy = 0

for split in ["train", "val", "test"]:
    split_dir = OUTPUT_ROOT / split
    if not split_dir.exists():
        continue
    for gloss_dir in sorted(split_dir.iterdir()):
        if not gloss_dir.is_dir():
            continue
        gloss = gloss_dir.name
        count = len(list(gloss_dir.glob("*.npy")))
        total_npy += count
        if gloss not in per_gloss_counts:
            per_gloss_counts[gloss] = {"train": 0, "val": 0, "test": 0, "total": 0}
        per_gloss_counts[gloss][split] = count
        per_gloss_counts[gloss]["total"] += count

# Build gloss mapping (sorted alphabetically)
glosses_sorted = sorted(per_gloss_counts.keys())
gloss_to_idx = {g: i for i, g in enumerate(glosses_sorted)}
idx_to_gloss = {str(i): g for i, g in enumerate(glosses_sorted)}

# Also keep the original WLASL class index mapping
original_class_mapping = {}
for idx, gloss in class_list.items():
    if gloss in per_gloss_counts:
        original_class_mapping[gloss] = idx

metadata = {
    "name": "wlasl2000_landmarks",
    "source": "wlasl2000",
    "description": "MediaPipe Hand Landmarks from full WLASL2000 dataset",
    "landmark_shape": [21, 3],
    "max_frames": MAX_FRAMES,
    "min_detection_rate": MIN_DETECTION_RATE,
    "normalization": "wrist_centered_scale_normalized",
    "interpolation": "linear",
    "statistics": {
        "total_processed": global_stats["total"],
        "total_success": global_stats["success"],
        "total_failed": global_stats["failed"],
        "success_rate": global_stats["success"] / max(global_stats["total"], 1),
        "mean_detection_rate": float(np.mean(global_stats["detection_rates"]))
            if global_stats["detection_rates"] else 0.0,
        "by_split": global_stats["by_split"],
        "total_npy_files": total_npy,
    },
    "num_classes": len(glosses_sorted),
    "glosses": glosses_sorted,
    "gloss_to_idx": gloss_to_idx,
    "idx_to_gloss": idx_to_gloss,
    "original_wlasl_class_mapping": original_class_mapping,
    "per_gloss_counts": per_gloss_counts,
    "errors": global_stats["errors"],
}

with open(OUTPUT_ROOT / "metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Metadata saved to: {OUTPUT_ROOT / 'metadata.json'}")
print(f"\nDataset summary:")
print(f"  Classes: {len(glosses_sorted)}")
print(f"  Total .npy files: {total_npy}")
print(f"  Train: {sum(g['train'] for g in per_gloss_counts.values())}")
print(f"  Val:   {sum(g['val'] for g in per_gloss_counts.values())}")
print(f"  Test:  {sum(g['test'] for g in per_gloss_counts.values())}")

Metadata saved to: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl2000/metadata.json

Dataset summary:
  Classes: 2000
  Total .npy files: 11844
  Train: 8210
  Val:   2232
  Test:  1402


In [9]:
# === 9. Verification ===

import subprocess

print("=" * 60)
print("VERIFICATION")
print("=" * 60)

# Count files per split
for split in ["train", "val", "test"]:
    split_dir = OUTPUT_ROOT / split
    if not split_dir.exists():
        print(f"{split}: directory not found")
        continue
    npy_files = list(split_dir.glob("**/*.npy"))
    gloss_dirs = [d for d in split_dir.iterdir() if d.is_dir()]
    print(f"{split}: {len(npy_files)} files across {len(gloss_dirs)} glosses")

# Load and verify random samples
import random
all_npy = list(OUTPUT_ROOT.glob("**/[!_]*.npy"))  # Exclude _progress files
if all_npy:
    samples = random.sample(all_npy, min(5, len(all_npy)))
    print(f"\nRandom sample verification:")
    for sample_path in samples:
        data = np.load(sample_path)
        rel = sample_path.relative_to(OUTPUT_ROOT)
        print(f"  {rel}: shape={data.shape}, "
              f"dtype={data.dtype}, range=[{data.min():.3f}, {data.max():.3f}]")

# Disk usage
result = subprocess.run(
    ['du', '-sh', str(OUTPUT_ROOT)],
    capture_output=True, text=True
)
print(f"\nTotal disk usage: {result.stdout.strip().split()[0]}")

VERIFICATION
train: 8210 files across 1998 glosses
val: 2232 files across 1691 glosses
test: 1402 files across 1180 glosses

Random sample verification:
  val/dive/16891.npy: shape=(29, 21, 3), dtype=float64, range=[-1.621, 0.830]
  train/underwear/60579.npy: shape=(32, 21, 3), dtype=float64, range=[-0.667, 0.987]
  train/embarrass/18897.npy: shape=(49, 21, 3), dtype=float64, range=[-1.323, 0.642]
  train/contribute/13026.npy: shape=(28, 21, 3), dtype=float64, range=[-0.967, 0.976]
  train/hold/27661.npy: shape=(31, 21, 3), dtype=float64, range=[-1.472, 0.917]

Total disk usage: 347M


In [10]:
# === 10. Cleanup ===

# Release MediaPipe resources
detector.close()
print("MediaPipe resources released.")

# Note about progress files
print(f"\nProgress files (safe to delete after verification):")
print(f"  {PROGRESS_FILE}")
print(f"  {ERROR_LOG_FILE}")
print("These files are only needed for resume capability.")

MediaPipe resources released.

Progress files (safe to delete after verification):
  /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl2000/_progress.json
  /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl2000/_errors.json
These files are only needed for resume capability.


---
## Summary

### Output Structure
```
06_datasets/wlasl/landmarks/wlasl2000/
├── metadata.json           (dataset metadata, class mappings)
├── _progress.json          (resume checkpoint - can delete when done)
├── _errors.json            (error log - keep for reference)
├── train/
│   ├── book/
│   │   ├── 69241.npy
│   │   └── ...
│   └── ...
├── val/
│   └── ...
└── test/
    └── ...
```

### Format
- Each `.npy` file: shape `(T, 21, 3)` where T <= 64
- 21 MediaPipe hand landmarks, 3 coordinates (x, y, z)
- Normalized: wrist-centered, scaled by wrist-to-middle-finger distance
- Missing frames interpolated linearly

### Next Steps
- Use `metadata.json` to create training subsets (filter by class count)
- Progressive training: start with top-100 glosses, scale to 300, 1000, 2000
- Filter glosses by minimum video count (e.g., >= 5 videos) for reliable training